# Regression Modeling

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore")

In [2]:
housing = pd.read_csv("../data/raw/housing.csv")
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
housing.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

### Why fit the imputer on the training data?

For a fair estimate of how a model will perform on unseen data, split the data before learning preprocessing values. Calculate the median for each numeric feature from the training set only, then use that same training-set median to fill missing values in both the training and test sets. Calculating the median from the entire dataset would let information from the test set influence preprocessing; this is a form of data leakage and can make evaluation less reliable. In cross-validation, the imputer should be fitted separately within each training fold, typically by placing it in a scikit-learn `Pipeline`.

The test set is still imputed—it is transformed using the median learned from training, rather than a separately calculated test median. Median imputation is a reasonable choice for numeric features when a simple fill is needed, especially when values may be skewed or affected by outliers, but it is not mandatory for every dataset or model. Choose an appropriate strategy based on the feature distributions and modeling needs. If you are only doing descriptive analysis and not evaluating a model on held-out data, using all available rows to calculate a summary statistic may be acceptable; it is the predictive evaluation workflow that requires fitting preprocessing on training data only.

In [4]:
# Reproduce the feature engineered variables from the previous notebook

housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]
housing["income_x_rooms_per_household"] = housing["median_income"] * housing["rooms_per_household"]

These features represent:

- rooms_per_household: average number of rooms per household.
- bedrooms_per_room: proportion of rooms that are bedrooms.
- population_per_household: average population per household.
- income_x_rooms_per_household: interaction between median income and average rooms per household.

The interaction feature allows the model to represent a combined effect of income and housing size, rather than treating them as entirely independent.

### Split the data

In [5]:
X = housing.drop(columns=["median_house_value"])
y = housing["median_house_value"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (16512, 13)
Testing features: (4128, 13)


## Build Preprocessing Pipeline

### Identify feature types

In [7]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'income_x_rooms_per_household']
Categorical features: ['ocean_proximity']


### Numerical Processing

What happens here?

1. SimpleImputer calculates the median of each numerical feature using training data only.
2. Missing values in the training and test sets are filled using those training medians.
3. StandardScaler learns the training means and standard deviations.
4. Both training and test values are transformed using those same learned parameters.

This is how we prevent preprocessing leakage.

In [8]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

### Categorical Preprocessing

Why One-Hot Encoding?

ocean_proximity contains categories such as INLAND, NEAR BAY, and NEAR OCEAN. Assigning arbitrary numbers to these categories could incorrectly imply that one category is numerically greater than another.

One-Hot Encoding creates separate binary columns instead.

In [9]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

### Combine the two pipelines

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)